# Fleet Compiler v0.5.1.1 — Many Adapters, No Training

This is the repaired v0.5 notebook.

### v0.5.1 fixes

- restores the missing `inspect_adapter()` preflight helpers,
- defines cache/output paths before preflight,
- makes license detection robust to different Hugging Face metadata shapes,
- verifies all required helper functions exist before scanning repos,
- treats 10+ experts as a **target**, not a fatal requirement,
- continues when individual Hub repos are dead/gated/incompatible,
- still fails if fewer than 4 usable experts survive,
- retains all v0.5 semantic-key, rank-retention, license, and routing guards.

No source-adapter training is performed.

## 0. Configuration

In [ ]:
BASE_MODEL = "Qwen/Qwen3-4B-Instruct-2507"

# ------------------------------------------------------------------
# Canonical Fleet ABI
# ------------------------------------------------------------------
CANONICAL_RANK = 32
MIN_STRUCTURAL_RETENTION = 0.95
DROP_LOW_RETENTION_EXPERTS = True

# Keep the T4 experiment bounded even if every fallback works.
MAX_ACTIVE_EXPERTS = 14
TARGET_EXPERTS = 10
MIN_EXPERTS = 4

TOP_K = 2
ROUTER_TEMPERATURE = 1.0
SEED = 42

# No repeated Qwen reloads unless explicitly requested.
RUN_FIDELITY = False
RUN_GENERATION = True

# ------------------------------------------------------------------
# Publishing / provenance
# ------------------------------------------------------------------
ALLOW_UNKNOWN_LICENSE_LOCAL = True

ALLOWED_REDIS_LICENSES = {
    "apache-2.0",
    "mit",
    "bsd-2-clause",
    "bsd-3-clause",
    "cc-by-4.0",
}

# ------------------------------------------------------------------
# Capability bank
#
# Order matters: high-value specialists are admitted first.
# Each capability has fallbacks, not duplicates in the active bank.
# ------------------------------------------------------------------
EXPERT_SLOTS = [
    {
        "name": "math",
        "description": "Mathematical reasoning and tutoring",
        "candidates": [
            "CanisAI/teach-math-qwen3-4b-2507-r1",
            "sinhal/qwen3-4b-instruct-2507-gsm8k-reasoning",
        ],
    },
    {
        "name": "science",
        "description": "Science tutoring and explanations",
        "candidates": [
            "CanisAI/teach-science-qwen3-4b-2507-r1",
        ],
    },
    {
        "name": "humanities",
        "description": "Humanities tutoring and explanations",
        "candidates": [
            "CanisAI/teach-humanities-qwen3-4b-2507-r1",
        ],
    },
    {
        "name": "language",
        "description": "Language tutoring and language-focused instruction",
        "candidates": [
            "CanisAI/teach-language-qwen3-4b-2507-r1",
        ],
    },
    {
        "name": "medical",
        "description": "Medical reasoning research/education specialist",
        "candidates": [
            "hxrikp/qwen3-4b-medical-reasoning-qlora",
            "towardsinnovationlab/Qwen3-4B_Instruct-medical",
        ],
    },
    {
        "name": "legal_ops",
        "description": "Narrow contract/legal-operations intake and extraction",
        "candidates": [
            "narcolepticchicken/qwen3-4b-legal-ops-contract-intake-lora",
        ],
    },
    {
        "name": "intent",
        "description": "Conversation intent classification",
        "candidates": [
            "lituokobe/Qwen3-4B-Instruct-2507-LoRA-Intent-Classifier",
        ],
    },
    {
        "name": "agent",
        "description": "Multi-turn agent trajectories and tool-use behavior",
        "candidates": [
            "yokobo-ai/qwen3-4b-agent-trajectory-lora-v6",
            "wan-wan/test03",
            "cjkasbdkjnlakb/agent-0925",
        ],
    },
    {
        "name": "function_calling",
        "description": "Function/tool selection and argument generation",
        "candidates": [
            "sweatSmile/Qwen3-4B-Function-Calling-Pro",
        ],
    },
    {
        "name": "structured",
        "description": "JSON/YAML/XML/TOML/CSV structured output",
        "candidates": [
            "matsunya/Qwen3_4B_Instruct_2507",
            "kimura0000/qwen3-4b-structured-output-lora-rev04",
            "marknchan/qwen3-4b-structured-output-lora",
        ],
    },
    {
        "name": "summary",
        "description": "Summarization and compression",
        "candidates": [
            "ryanu/Qwen3-summary-lora",
        ],
    },
    {
        "name": "prompt",
        "description": "Prompt/instruction behavior",
        "candidates": [
            "Limbicnation/qwen3-4b-prompt-lora",
        ],
    },
    {
        "name": "generalist",
        "description": "General tutoring behavior",
        "candidates": [
            "CanisAI/teach-generalist-qwen3-4b-2507-r1",
        ],
    },
    {
        "name": "recursive",
        "description": "Recursive RLVR reasoning stress expert",
        "candidates": [
            "lsteno/Qwen3-4B-Instruct-2507-RLM-RLVR-depth2-recursive-r64-a128-lr1e-5-adapter",
        ],
    },
]

print({
    "base": BASE_MODEL,
    "canonical_rank": CANONICAL_RANK,
    "retention_floor": MIN_STRUCTURAL_RETENTION,
    "top_k": TOP_K,
    "max_active_experts": MAX_ACTIVE_EXPERTS,
    "target_experts": TARGET_EXPERTS,
    "minimum_experts": MIN_EXPERTS,
    "candidate_slots": len(EXPERT_SLOTS),
    "training": False,
})

## 1. Install

In [ ]:
%pip install -q -U \
    "peft>=0.20.0" \
    transformers \
    accelerate \
    bitsandbytes \
    safetensors \
    huggingface_hub

In [ ]:
import gc
import json
import math
import re
import shutil
import types
from collections import defaultdict
from pathlib import Path

import pandas as pd
import torch
import torch.nn.functional as F

import peft
import transformers

from huggingface_hub import (
    HfApi,
    hf_hub_download,
    snapshot_download,
)
from peft import (
    ArrowConfig,
    LoraConfig,
    PeftModel,
    TaskType,
    create_arrow_model,
)
from safetensors.torch import load_file, save_file
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)

print("Python:", __import__("sys").version.split()[0])
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("peft:", peft.__version__)

if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime required for final inference.")

print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

assert hasattr(peft, "ArrowConfig")
assert hasattr(peft, "create_arrow_model")

## 2. Inspect candidate adapters first

Only tiny configs/metadata are read first. Fleet rejects unsupported LoRA variants rather than silently miscompiling them.

License policy is separate from local testing:
- `redistributable=True` → compiled derivative may enter the Hub-ready folder.
- unknown license → local test is allowed by default, but compiled derivative is not prepared for upload.

In [ ]:
# ------------------------------------------------------------------
# PRE-FLIGHT HELPERS
# This cell is intentionally self-contained so selection cannot run
# before its dependencies exist.
# ------------------------------------------------------------------

CACHE_ROOT = Path("/content/fleet_v0511")
SOURCE_ROOT = CACHE_ROOT / "sources"
COMPILED_ROOT = CACHE_ROOT / "compiled"
PACKAGE_ROOT = CACHE_ROOT / "packages"

for path in [
    SOURCE_ROOT,
    COMPILED_ROOT,
    PACKAGE_ROOT,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

api = HfApi()


def canonical_base_family(value):
    if not value:
        return None

    value = (
        str(value)
        .lower()
        .replace("\\", "/")
        .rstrip("/")
    )

    # Accept official Qwen, Unsloth wrappers, and local paths that
    # clearly identify the same Qwen3-4B-Instruct-2507 family.
    if "qwen3-4b-instruct-2507" in value:
        return "qwen3-4b-instruct-2507"

    return value.split("/")[-1]


EXPECTED_FAMILY = canonical_base_family(
    BASE_MODEL
)


def _read_card_license(card_data):
    if card_data is None:
        return None

    # huggingface_hub versions may return an object or dict-like card.
    if isinstance(card_data, dict):
        value = card_data.get("license")
    else:
        value = getattr(
            card_data,
            "license",
            None,
        )

    if isinstance(value, (list, tuple)):
        value = value[0] if value else None

    return (
        str(value).lower()
        if value
        else None
    )


def get_repo_license(repo_id):
    try:
        info = api.model_info(
            repo_id
        )

        license_id = _read_card_license(
            getattr(
                info,
                "card_data",
                None,
            )
        )

        if not license_id:
            for tag in (
                getattr(
                    info,
                    "tags",
                    None,
                )
                or []
            ):
                tag = str(tag)

                if tag.startswith(
                    "license:"
                ):
                    license_id = (
                        tag.split(
                            ":",
                            1,
                        )[1]
                        .strip()
                        .lower()
                    )
                    break

        return license_id

    except Exception as e:
        # License metadata failure should NOT make an otherwise valid
        # local experiment fail. Unknown-license artifacts just won't
        # enter the Hub-ready derivative folder.
        print(
            f"[license warning] "
            f"{repo_id}: "
            f"{type(e).__name__}: {e}"
        )
        return None


def inspect_adapter(repo_id):
    # Download only the tiny config here.
    cfg_path = hf_hub_download(
        repo_id=repo_id,
        filename="adapter_config.json",
    )

    cfg = json.loads(
        Path(cfg_path).read_text(
            encoding="utf-8"
        )
    )

    problems = []

    peft_type = str(
        cfg.get(
            "peft_type",
            "",
        )
    ).upper()

    if peft_type != "LORA":
        problems.append(
            f"peft_type={peft_type or None}"
        )

    family = canonical_base_family(
        cfg.get(
            "base_model_name_or_path"
        )
    )

    if family != EXPECTED_FAMILY:
        problems.append(
            f"base family={family}"
        )

    # v0.5.1 compiler math only supports ordinary LoRA.
    if cfg.get("use_dora", False):
        problems.append("DoRA")

    if cfg.get("bias", "none") != "none":
        problems.append(
            f"bias={cfg.get('bias')}"
        )

    if cfg.get("modules_to_save"):
        problems.append(
            "modules_to_save"
        )

    # Empty dicts are fine; non-empty per-module overrides are not.
    if cfg.get("rank_pattern"):
        problems.append(
            "rank_pattern"
        )

    if cfg.get("alpha_pattern"):
        problems.append(
            "alpha_pattern"
        )

    if (
        cfg.get(
            "layers_to_transform"
        )
        is not None
    ):
        problems.append(
            "layers_to_transform"
        )

    if cfg.get("target_parameters"):
        problems.append(
            "target_parameters"
        )

    if cfg.get(
        "fan_in_fan_out",
        False,
    ):
        problems.append(
            "fan_in_fan_out=True"
        )

    if cfg.get(
        "layer_replication"
    ):
        problems.append(
            "layer_replication"
        )

    targets = (
        cfg.get("target_modules")
        or []
    )

    if isinstance(
        targets,
        str,
    ):
        targets = [targets]

    license_id = get_repo_license(
        repo_id
    )

    return {
        "repo": repo_id,
        "config": cfg,
        "family": family,
        "rank": int(
            cfg.get("r", 0)
        ),
        "alpha": float(
            cfg.get(
                "lora_alpha",
                0,
            )
        ),
        "targets": sorted(
            str(x)
            for x in targets
        ),
        "license": license_id,
        "redistributable": (
            license_id
            in ALLOWED_REDIS_LICENSES
        ),
        "problems": problems,
    }


# Fail HERE with a useful message instead of turning every adapter
# into "NameError: inspect_adapter".
REQUIRED_PREFLIGHT_HELPERS = [
    "canonical_base_family",
    "get_repo_license",
    "inspect_adapter",
]

missing_helpers = [
    name
    for name in REQUIRED_PREFLIGHT_HELPERS
    if name not in globals()
]

if missing_helpers:
    raise RuntimeError(
        "Fleet notebook assembly error. "
        "Missing preflight helpers: "
        + ", ".join(
            missing_helpers
        )
    )

print(
    "✅ Preflight helpers ready."
)


# ------------------------------------------------------------------
# CAPABILITY SLOT SELECTION
# ------------------------------------------------------------------

inspection_rows = []
selected = []

for slot in EXPERT_SLOTS:
    valid_for_slot = []

    for repo_id in slot[
        "candidates"
    ]:
        try:
            info = inspect_adapter(
                repo_id
            )

            inspection_rows.append({
                "slot": slot["name"],
                "repo": repo_id,
                "valid": not info[
                    "problems"
                ],
                "rank": info["rank"],
                "targets": ", ".join(
                    info["targets"]
                ),
                "license": info[
                    "license"
                ],
                "redistributable": info[
                    "redistributable"
                ],
                "problems": "; ".join(
                    info["problems"]
                ),
            })

            if not info["problems"]:
                valid_for_slot.append({
                    "name": slot["name"],
                    "description": slot[
                        "description"
                    ],
                    **info,
                })

        except Exception as e:
            inspection_rows.append({
                "slot": slot["name"],
                "repo": repo_id,
                "valid": False,
                "rank": None,
                "targets": "",
                "license": None,
                "redistributable": False,
                "problems": (
                    f"{type(e).__name__}: "
                    f"{e}"
                ),
            })

    if not valid_for_slot:
        print(
            f"[skip] no valid adapter "
            f"for capability "
            f"{slot['name']!r}"
        )
        continue

    # Within one capability, prefer redistributable sources,
    # then the smaller rank to reduce compilation/runtime cost.
    valid_for_slot.sort(
        key=lambda x: (
            not x[
                "redistributable"
            ],
            x["rank"],
        )
    )

    chosen = valid_for_slot[0]

    if (
        chosen["license"]
        is None
        and not
        ALLOW_UNKNOWN_LICENSE_LOCAL
    ):
        print(
            "[skip] unknown license:",
            chosen["repo"],
        )
        continue

    selected.append(
        chosen
    )


# Keep all capability slots up to our runtime cap.
if len(selected) > MAX_ACTIVE_EXPERTS:
    overflow = selected[
        MAX_ACTIVE_EXPERTS:
    ]

    selected = selected[
        :MAX_ACTIVE_EXPERTS
    ]

    print(
        "\\nNot downloaded because "
        "MAX_ACTIVE_EXPERTS was reached:"
    )

    for item in overflow:
        print(
            " -",
            item["name"],
            item["repo"],
        )


inspection_df = pd.DataFrame(
    inspection_rows
)

print("\\nPREFLIGHT")

if len(inspection_df):
    print(
        inspection_df.to_string(
            index=False
        )
    )
else:
    print(
        "No candidates were inspected."
    )


print("\\nSELECTED FOR DOWNLOAD")

for item in selected:
    print(
        f"- {item['name']:16} "
        f"r={item['rank']:<3} "
        f"license={item['license']!s:<12} "
        f"targets={item['targets']} "
        f"repo={item['repo']}"
    )


print(
    f"\\nSelected {len(selected)} "
    f"experts. Target={TARGET_EXPERTS}, "
    f"minimum={MIN_EXPERTS}."
)

if len(selected) < TARGET_EXPERTS:
    print(
        "⚠️ Fewer than the target expert count survived "
        "preflight. Fleet will continue with the valid bank."
    )

if len(selected) < MIN_EXPERTS:
    raise RuntimeError(
        f"Only {len(selected)} valid experts found. "
        f"Fleet needs at least {MIN_EXPERTS} "
        f"for a useful routing experiment."
    )

## 3. Download only the selected adapter files

In [ ]:
def download_adapter(item):
    local = Path(snapshot_download(
        repo_id=item["repo"],
        allow_patterns=[
            "adapter_config.json",
            "adapter_model.safetensors",
            "adapter_model.bin",
            "README.md",
            "LICENSE",
            "LICENSE*",
        ],
    ))

    if not (
        (local / "adapter_model.safetensors").exists()
        or (local / "adapter_model.bin").exists()
    ):
        raise FileNotFoundError(
            f"No adapter weights in {item['repo']}"
        )

    return local

SOURCE_PATHS = {}

for item in selected:
    print("Downloading:", item["repo"])
    path = download_adapter(item)
    SOURCE_PATHS[item["name"]] = path

    size_mb = sum(
        f.stat().st_size
        for f in path.glob("adapter_model.*")
        if f.is_file()
    ) / 1024**2

    print(f"  {size_mb:.2f} MB")

## 4. Canonicalize third-party LoRA module keys

This is the important v0.4 fix.

Different tools can save the same Qwen layer under different wrapper prefixes. Fleet now ignores those wrappers and reduces every LoRA key to a semantic identity:

```text
layers.<layer_number>.<self_attn|mlp>.<target>
```

Example:

```text
base_model.model.model.layers.0.self_attn.q_proj
base_model.model.model.model.layers.0.self_attn.q_proj
```

both become:

```text
layers.0.self_attn.q_proj
```

In [ ]:
A_SUFFIX = ".lora_A.weight"
B_SUFFIX = ".lora_B.weight"

SEMANTIC_RE = re.compile(
    r"(?:^|\.)layers\.(\d+)\.(self_attn|mlp)\."
    r"(q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj)"
    r"(?:\.base_layer)?$"
)

def semantic_module_id(prefix):
    match = SEMANTIC_RE.search(prefix)

    if not match:
        raise ValueError(
            f"Cannot canonicalize LoRA path: {prefix}"
        )

    layer, block, target = match.groups()
    return f"layers.{int(layer)}.{block}.{target}"

def storage_prefix(semantic_id):
    # Standard PEFT path for Qwen3ForCausalLM adapter checkpoints.
    return f"base_model.model.model.{semantic_id}"

def load_adapter_state(path):
    path = Path(path)
    safe = path / "adapter_model.safetensors"
    legacy = path / "adapter_model.bin"

    if safe.exists():
        return load_file(str(safe), device="cpu")

    if legacy.exists():
        state = torch.load(
            legacy,
            map_location="cpu",
            weights_only=True,
        )
        if isinstance(state, dict):
            return state

    raise FileNotFoundError(f"No supported adapter weights in {path}")

def load_pairs(path):
    state = load_adapter_state(path)
    pairs = {}

    for key, tensor in state.items():
        if not key.endswith(A_SUFFIX):
            continue

        raw_prefix = key[:-len(A_SUFFIX)]
        b_key = raw_prefix + B_SUFFIX

        if b_key not in state:
            raise KeyError(f"Missing B for {raw_prefix}")

        semantic_id = semantic_module_id(raw_prefix)

        if semantic_id in pairs:
            raise ValueError(
                f"Duplicate semantic module: {semantic_id}"
            )

        pairs[semantic_id] = {
            "A": tensor,
            "B": state[b_key],
            "raw_prefix": raw_prefix,
        }

    if not pairs:
        raise RuntimeError(f"No standard LoRA pairs in {path}")

    return pairs

SOURCE_CONFIGS = {}
SOURCE_PAIRS = {}

for item in selected:
    name = item["name"]
    path = SOURCE_PATHS[name]

    cfg = json.loads(
        (path / "adapter_config.json").read_text(encoding="utf-8")
    )

    pairs = load_pairs(path)

    SOURCE_CONFIGS[name] = cfg
    SOURCE_PAIRS[name] = pairs

    sample_id, sample = next(iter(pairs.items()))

    print(
        f"{name:12} r={cfg['r']:<3} "
        f"modules={len(pairs):<3} "
        f"sample={sample_id} <- {sample['raw_prefix']}"
    )

## 5. Build the semantic union schema + origin masks

In [ ]:
def build_schema(source_pairs):
    schema = {}

    for expert, pairs in source_pairs.items():
        for semantic_id, pair in pairs.items():
            shape = (
                pair["B"].shape[0],
                pair["A"].shape[1],
            )

            if (
                semantic_id in schema
                and schema[semantic_id] != shape
            ):
                raise ValueError(
                    f"Shape conflict at {semantic_id}"
                )

            schema[semantic_id] = shape

    return schema

def rebuild_bank_metadata():
    global MODULE_SCHEMA, LEAF_TARGETS, LAYER_IDS, ORIGIN_MASKS

    MODULE_SCHEMA = build_schema(SOURCE_PAIRS)

    LEAF_TARGETS = sorted({
        key.split(".")[-1]
        for key in MODULE_SCHEMA
    })

    LAYER_IDS = sorted({
        int(key.split(".")[1])
        for key in MODULE_SCHEMA
    })

    ORIGIN_MASKS = {
        expert: {
            key: key in SOURCE_PAIRS[expert]
            for key in MODULE_SCHEMA
        }
        for expert in SOURCE_PAIRS
    }

    theoretical_max = len(LAYER_IDS) * len(LEAF_TARGETS)

    if len(MODULE_SCHEMA) > theoretical_max:
        raise RuntimeError(
            "Duplicate module namespaces survived semantic normalization."
        )

    print(
        f"layers={len(LAYER_IDS)}, "
        f"targets={LEAF_TARGETS}, "
        f"slots={len(MODULE_SCHEMA)}"
    )

    for expert, mask in ORIGIN_MASKS.items():
        active = int(sum(mask.values()))

        if active == 0:
            raise RuntimeError(
                f"{expert} has zero modules after normalization."
            )

        print(
            f"{expert:12} active={active}/{len(mask)} "
            f"({100 * active / len(mask):.1f}%)"
        )

rebuild_bank_metadata()

print("\n✅ Semantic module normalization passed.")

## 6. Low-rank compiler math

In [ ]:
def source_scaling(cfg):
    r = int(cfg["r"])
    alpha = float(cfg["lora_alpha"])

    if cfg.get("use_rslora", False):
        return alpha / math.sqrt(r)

    return alpha / r

def refactor_lora_delta(
    A,
    B,
    source_scale,
    target_rank,
):
    A = A.float()
    B = B.float()

    source_rank = A.shape[0]
    out_dim = B.shape[0]
    in_dim = A.shape[1]

    if target_rank >= source_rank:
        mag = math.sqrt(abs(float(source_scale)))
        sign = 1.0 if source_scale >= 0 else -1.0

        A_out = torch.zeros(
            target_rank,
            in_dim,
            dtype=torch.float32,
        )
        B_out = torch.zeros(
            out_dim,
            target_rank,
            dtype=torch.float32,
        )

        A_out[:source_rank] = A * mag
        B_out[:, :source_rank] = B * (mag * sign)

        return A_out, B_out, {
            "source_rank": source_rank,
            "target_rank": target_rank,
            "retained_energy": 1.0,
            "mode": (
                "exact"
                if source_rank == target_rank
                else "exact_pad"
            ),
        }

    # Work in the small source-rank core.
    Qb, Rb = torch.linalg.qr(B, mode="reduced")
    Qa, Ra = torch.linalg.qr(A.T, mode="reduced")

    core = float(source_scale) * (Rb @ Ra.T)
    U, S, Vh = torch.linalg.svd(core, full_matrices=False)

    k = min(target_rank, S.numel())
    sqrt_s = torch.sqrt(torch.clamp(S[:k], min=0))

    Bk = (Qb @ U[:, :k]) * sqrt_s.unsqueeze(0)
    Ak = sqrt_s.unsqueeze(1) * (Vh[:k, :] @ Qa.T)

    A_out = torch.zeros(
        target_rank,
        in_dim,
        dtype=torch.float32,
    )
    B_out = torch.zeros(
        out_dim,
        target_rank,
        dtype=torch.float32,
    )

    A_out[:k] = Ak
    B_out[:, :k] = Bk

    total = float(torch.sum(S**2).item())
    kept = float(torch.sum(S[:k]**2).item())

    return A_out, B_out, {
        "source_rank": source_rank,
        "target_rank": target_rank,
        "retained_energy": (
            1.0 if total == 0 else kept / total
        ),
        "mode": "truncate_svd",
    }

# Exact-expansion sanity check.
torch.manual_seed(SEED)
A = torch.randn(8, 37)
B = torch.randn(41, 8)

Ac, Bc, _ = refactor_lora_delta(
    A, B, source_scale=2.0, target_rank=16
)

rel_error = (
    torch.linalg.norm(2.0 * (B @ A) - (Bc @ Ac))
    / torch.linalg.norm(2.0 * (B @ A))
).item()

print("exact expansion relative error:", rel_error)
assert rel_error < 1e-5

print("✅ Compiler math passed.")

## 7. Rank sweep + structural-retention gate

A high-rank adapter is not automatically forced into the bank.

If its update loses too much singular energy at the chosen canonical rank, Fleet drops it from the active bank and rebuilds the schema.

In [ ]:
def lowrank_singular_values(A, B, source_scale):
    A = A.float()
    B = B.float()

    Qb, Rb = torch.linalg.qr(B, mode="reduced")
    Qa, Ra = torch.linalg.qr(A.T, mode="reduced")

    core = float(source_scale) * (Rb @ Ra.T)
    return torch.linalg.svdvals(core)

def retention_for_expert(expert, target_rank):
    cfg = SOURCE_CONFIGS[expert]
    scale = source_scaling(cfg)

    total = 0.0
    kept = 0.0

    for pair in SOURCE_PAIRS[expert].values():
        s = lowrank_singular_values(
            pair["A"],
            pair["B"],
            scale,
        )

        energy = s.square()
        total += float(energy.sum().item())

        k = min(target_rank, s.numel())
        kept += float(energy[:k].sum().item())

    return 1.0 if total == 0 else kept / total

RANK_CANDIDATES = [4, 8, 12, 16, 24, 32, 40, 48, 56, 64]

rows = []

for rank in RANK_CANDIDATES:
    ret = {
        expert: retention_for_expert(expert, rank)
        for expert in SOURCE_PAIRS
    }

    rows.append({
        "rank": rank,
        "worst_retention": min(ret.values()),
        "mean_retention": sum(ret.values()) / len(ret),
        **{
            f"{expert}_retention": value
            for expert, value in ret.items()
        },
    })

rank_sweep_df = pd.DataFrame(rows)
print(rank_sweep_df.to_string(index=False))

retention_at_rank = {
    expert: retention_for_expert(expert, CANONICAL_RANK)
    for expert in SOURCE_PAIRS
}

print("\nRETENTION AT CANONICAL RANK")
for expert, value in retention_at_rank.items():
    print(f"{expert:12} {value:.4%}")

dropped_for_retention = []

if DROP_LOW_RETENTION_EXPERTS:
    dropped_for_retention = [
        expert
        for expert, value in retention_at_rank.items()
        if value < MIN_STRUCTURAL_RETENTION
    ]

    if dropped_for_retention:
        print(
            "\nDropping low-retention experts:",
            dropped_for_retention,
        )

        for expert in dropped_for_retention:
            SOURCE_PAIRS.pop(expert, None)
            SOURCE_CONFIGS.pop(expert, None)
            SOURCE_PATHS.pop(expert, None)

        selected = [
            item
            for item in selected
            if item["name"] not in dropped_for_retention
        ]

        if len(SOURCE_PAIRS) < MIN_EXPERTS:
            raise RuntimeError(
                f"Only {len(SOURCE_PAIRS)} experts remain "
                f"after retention gating."
            )

        rebuild_bank_metadata()


# Keep the runtime bounded if future edits add even more slots.
if len(SOURCE_PAIRS) > MAX_ACTIVE_EXPERTS:
    keep = list(SOURCE_PAIRS)[:MAX_ACTIVE_EXPERTS]
    drop_capacity = [
        expert
        for expert in SOURCE_PAIRS
        if expert not in keep
    ]

    for expert in drop_capacity:
        SOURCE_PAIRS.pop(expert, None)
        SOURCE_CONFIGS.pop(expert, None)
        SOURCE_PATHS.pop(expert, None)

    selected = [
        item
        for item in selected
        if item["name"] in SOURCE_PAIRS
    ]

    rebuild_bank_metadata()

    print(
        "\nDropped by MAX_ACTIVE_EXPERTS capacity:",
        drop_capacity,
    )

if len(SOURCE_PAIRS) < MIN_EXPERTS:
    raise RuntimeError(
        f"Only {len(SOURCE_PAIRS)} experts remain after all gates; "
        f"need at least {MIN_EXPERTS}."
    )

print("\nACTIVE BANK:", list(SOURCE_PAIRS))
print("ACTIVE EXPERT COUNT:", len(SOURCE_PAIRS))

if len(SOURCE_PAIRS) < TARGET_EXPERTS:
    print(
        f"⚠️ Active bank has {len(SOURCE_PAIRS)} experts, "
        f"below target {TARGET_EXPERTS}, but above the "
        f"minimum {MIN_EXPERTS}; continuing."
    )

## 8. Compile active experts into one canonical PEFT ABI

In [ ]:
def compile_expert(expert):
    cfg = SOURCE_CONFIGS[expert]
    pairs = SOURCE_PAIRS[expert]

    out_dir = COMPILED_ROOT / expert

    if out_dir.exists():
        shutil.rmtree(out_dir)

    out_dir.mkdir(parents=True)

    scale = source_scaling(cfg)
    state = {}
    module_report = []
    origin_mask = {}

    for semantic_id, (out_dim, in_dim) in MODULE_SCHEMA.items():
        existed = semantic_id in pairs
        origin_mask[semantic_id] = existed

        if existed:
            pair = pairs[semantic_id]

            A_new, B_new, info = refactor_lora_delta(
                pair["A"],
                pair["B"],
                source_scale=scale,
                target_rank=CANONICAL_RANK,
            )
        else:
            A_new = torch.zeros(
                CANONICAL_RANK,
                in_dim,
                dtype=torch.float32,
            )
            B_new = torch.zeros(
                out_dim,
                CANONICAL_RANK,
                dtype=torch.float32,
            )
            info = {
                "source_rank": 0,
                "target_rank": CANONICAL_RANK,
                "retained_energy": 1.0,
                "mode": "zero_slot",
            }

        state[
            storage_prefix(semantic_id) + A_SUFFIX
        ] = A_new.contiguous()

        state[
            storage_prefix(semantic_id) + B_SUFFIX
        ] = B_new.contiguous()

        module_report.append({
            "module": semantic_id,
            "origin_active": existed,
            **info,
        })

    save_file(
        state,
        str(out_dir / "adapter_model.safetensors"),
    )

    compiled_cfg = LoraConfig(
        r=CANONICAL_RANK,
        lora_alpha=CANONICAL_RANK,
        lora_dropout=0.0,
        target_modules=LEAF_TARGETS,
        bias="none",
        task_type=TaskType.CAUSAL_LM,
        inference_mode=True,
    )

    compiled_cfg.base_model_name_or_path = BASE_MODEL
    compiled_cfg.save_pretrained(out_dir)

    (out_dir / "fleet_origin_mask.json").write_text(
        json.dumps(origin_mask, indent=2),
        encoding="utf-8",
    )

    real = [
        row
        for row in module_report
        if row["origin_active"]
    ]

    source_meta = next(
        item for item in selected
        if item["name"] == expert
    )

    report = {
        "expert": expert,
        "description": source_meta["description"],
        "source_repo": source_meta["repo"],
        "source_license": source_meta["license"],
        "redistributable": source_meta["redistributable"],
        "source_rank": int(cfg["r"]),
        "canonical_rank": CANONICAL_RANK,
        "source_targets": sorted(
            cfg["target_modules"]
            if isinstance(
                cfg["target_modules"],
                list,
            )
            else [cfg["target_modules"]]
        ),
        "canonical_targets": LEAF_TARGETS,
        "origin_active_modules": int(sum(origin_mask.values())),
        "canonical_slots": len(origin_mask),
        "mean_retained_energy": float(
            sum(x["retained_energy"] for x in real) / len(real)
        ),
        "min_retained_energy": float(
            min(x["retained_energy"] for x in real)
        ),
        "module_report": module_report,
    }

    (out_dir / "fleet_compile_report.json").write_text(
        json.dumps(report, indent=2),
        encoding="utf-8",
    )

    return report

COMPILE_REPORTS = {
    expert: compile_expert(expert)
    for expert in SOURCE_PAIRS
}

compile_df = pd.DataFrame([
    {
        "expert": expert,
        "source_rank": report["source_rank"],
        "canonical_rank": report["canonical_rank"],
        "active_modules": report["origin_active_modules"],
        "canonical_slots": report["canonical_slots"],
        "mean_energy": report["mean_retained_energy"],
        "min_energy": report["min_retained_energy"],
        "license": report["source_license"],
    }
    for expert, report in COMPILE_REPORTS.items()
])

print(compile_df.to_string(index=False))

## 9. Verify the canonical ABI

In [ ]:
compiled_rows = []

for expert in SOURCE_PAIRS:
    cfg = json.loads(
        (
            COMPILED_ROOT
            / expert
            / "adapter_config.json"
        ).read_text(encoding="utf-8")
    )

    targets = cfg["target_modules"]
    if isinstance(targets, str):
        targets = [targets]

    compiled_rows.append({
        "expert": expert,
        "rank": cfg["r"],
        "targets": ", ".join(sorted(targets)),
    })

compiled_df = pd.DataFrame(compiled_rows)
print(compiled_df.to_string(index=False))

assert len(set(compiled_df["rank"])) == 1
assert len(set(compiled_df["targets"])) == 1

print("\n✅ Active experts share one Arrow-compatible ABI.")

## 10. Package `.adapter` files with provenance

In [ ]:
PACKAGES = {}
PUBLISHABLE_PACKAGES = {}

selected_by_name = {
    item["name"]: item
    for item in selected
}

for expert in SOURCE_PAIRS:
    src = COMPILED_ROOT / expert
    stage = PACKAGE_ROOT / f"{expert}_stage"

    if stage.exists():
        shutil.rmtree(stage)

    shutil.copytree(src, stage)

    meta = selected_by_name[expert]

    manifest = {
        "format": "fleet.adapter",
        "format_version": 4,
        "name": expert,
        "description": meta["description"],
        "base_model": BASE_MODEL,
        "source_repo": meta["repo"],
        "source_license": meta["license"],
        "redistributable_by_policy": meta["redistributable"],
        "canonical_rank": CANONICAL_RANK,
        "canonical_targets": LEAF_TARGETS,
        "origin_mask": "fleet_origin_mask.json",
        "compiler": (
            "semantic-key normalization + "
            "thin-QR + core-SVD"
        ),
    }

    (stage / "manifest.json").write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )

    temp = shutil.make_archive(
        str(PACKAGE_ROOT / expert),
        "zip",
        root_dir=stage,
    )

    final = PACKAGE_ROOT / f"{expert}.adapter"

    if final.exists():
        final.unlink()

    Path(temp).rename(final)
    PACKAGES[expert] = final

    if meta["redistributable"]:
        PUBLISHABLE_PACKAGES[expert] = final

    print(
        f"{expert:12} "
        f"{final.stat().st_size / 1024**2:.2f} MB "
        f"publishable={meta['redistributable']} "
        f"license={meta['license']}"
    )

print("\nPublishable compiled experts:", list(PUBLISHABLE_PACKAGES))

## 11. Optional source-vs-compiled fidelity

This is **off by default** because it repeatedly loads Qwen.

Turn it on only when you want behavioral fidelity measurements.

In [ ]:
BF16_OK = torch.cuda.is_bf16_supported()
COMPUTE_DTYPE = (
    torch.bfloat16
    if BF16_OK
    else torch.float16
)

def bnb_config():
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    )

tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL,
    use_fast=True,
)

if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

FIDELITY_PROMPTS = {
    "math": [
        "If 8 pens cost $12, how much do 14 pens cost at the same rate?"
    ],
    "science": [
        "Explain why increasing temperature often speeds up a chemical reaction."
    ],
    "humanities": [
        "Explain three important causes of the French Revolution to a student."
    ],
    "language": [
        "Explain the difference between a noun and a verb with simple examples."
    ],
    "medical": [
        "For education only, explain the physiological role of insulin."
    ],
    "legal_ops": [
        'Extract the governing-law clause from: "This Agreement shall be governed by the laws of New York."'
    ],
    "intent": [
        'Classify the intent: "I need to move my appointment to Friday."'
    ],
    "agent": [
        "Plan how to inspect a Python repository and locate a failing API test."
    ],
    "function_calling": [
        'Choose a tool call for: "What is the weather in Manila?"'
    ],
    "structured": [
        'Return JSON with keys "name" and "value" for apple, 3.'
    ],
    "summary": [
        "Summarize: The city approved a new transit plan after months of public debate."
    ],
    "prompt": [
        "Answer in exactly three concise bullet points about recursion."
    ],
    "generalist": [
        "Teach a student how photosynthesis works."
    ],
    "recursive": [
        "Solve carefully: If x + y = 10 and xy = 21, what are x and y?"
    ],
}

fidelity_rows = []

if RUN_FIDELITY:
    for expert in SOURCE_PAIRS:
        print("Fidelity:", expert)

        base = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=bnb_config(),
            dtype=COMPUTE_DTYPE,
            device_map={"": 0},
        )

        model = PeftModel.from_pretrained(
            base,
            SOURCE_PATHS[expert],
            adapter_name="source",
            is_trainable=False,
        )

        model.load_adapter(
            COMPILED_ROOT / expert,
            adapter_name="compiled",
            is_trainable=False,
        )

        prompt = FIDELITY_PROMPTS.get(
            expert,
            ["Explain your specialization briefly."],
        )[0]

        inputs = tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

        device = next(model.parameters()).device
        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        with torch.inference_mode():
            model.set_adapter("source")
            src_logits = model(
                **inputs
            ).logits[:, -1, :].float()

            model.set_adapter("compiled")
            cmp_logits = model(
                **inputs
            ).logits[:, -1, :].float()

        fidelity_rows.append({
            "expert": expert,
            "kl": F.kl_div(
                F.log_softmax(cmp_logits, dim=-1),
                F.softmax(src_logits, dim=-1),
                reduction="batchmean",
            ).item(),
            "logit_mae": torch.mean(
                torch.abs(src_logits - cmp_logits)
            ).item(),
            "top1_agreement": float(
                (
                    src_logits.argmax(-1)
                    == cmp_logits.argmax(-1)
                ).float().mean().item()
            ),
        })

        del model, base
        gc.collect()
        torch.cuda.empty_cache()

fidelity_df = pd.DataFrame(fidelity_rows)

if len(fidelity_df):
    print(fidelity_df.to_string(index=False))
else:
    print("Fidelity check skipped.")

## 12. Build one final routed model

In [ ]:
gc.collect()
torch.cuda.empty_cache()

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config(),
    dtype=COMPUTE_DTYPE,
    device_map={"": 0},
)

base_model.config.use_cache = True

EXPERT_ORDER = list(SOURCE_PAIRS.keys())

compiled_paths = [
    str(COMPILED_ROOT / expert)
    for expert in EXPERT_ORDER
]

arrow_cfg = ArrowConfig(
    top_k=min(TOP_K, len(EXPERT_ORDER)),
    router_temperature=ROUTER_TEMPERATURE,
    rng_seed=SEED,
)

fleet_model = create_arrow_model(
    base_model=base_model,
    task_specific_adapter_paths=compiled_paths,
    arrow_config=arrow_cfg,
)

fleet_model.set_adapter("arrow_router")

print("model:", type(fleet_model))
print("adapters:", list(fleet_model.peft_config.keys()))
print("experts:", EXPERT_ORDER)

assert "arrow_router" in fleet_model.peft_config

print("\n✅ One routed PeftModel object built.")

## 13. Install Fleet's source-mask-aware internal router

This is not a prompt classifier.

The mask is applied **inside each LoRA layer before top-k**, so a compatibility zero-slot cannot consume an expert slot.

In [ ]:
def resolve_schema_key(host_name):
    try:
        return semantic_module_id(host_name)
    except Exception:
        return None

def fleet_masked_arrow_forward(
    self,
    x,
    lora_A,
    lora_B,
    dropout,
    scaling,
):
    x = self._cast_input_dtype(
        x,
        lora_A[self.task_adapter_names[0]].weight.dtype,
    )

    batch, *rest, in_features = x.shape
    tok = x.view(-1, in_features)

    valid_global = torch.nonzero(
        self.fleet_valid_expert_mask.to(tok.device),
        as_tuple=False,
    ).flatten()

    if valid_global.numel() == 0:
        out_features = lora_B[
            self.task_adapter_names[0]
        ].weight.shape[0]

        return tok.new_zeros(
            batch,
            *rest,
            out_features,
        )

    valid_names = [
        self.task_adapter_names[int(i)]
        for i in valid_global.tolist()
    ]

    valid_prototypes = self.prototypes.index_select(
        0,
        valid_global,
    )

    sim = torch.abs(
        tok @ valid_prototypes.T
    )

    effective_k = min(
        self.top_k,
        len(valid_names),
    )

    top_v, local_idx = torch.topk(
        sim,
        effective_k,
        dim=1,
    )

    full_score = tok.new_full(
        (tok.shape[0], len(valid_names)),
        float("-inf"),
    )

    full_score.scatter_(
        1,
        local_idx,
        top_v,
    )

    coeff = torch.softmax(
        full_score / self.temperature,
        dim=1,
    )

    scales = torch.tensor(
        [scaling[name] for name in valid_names],
        device=tok.device,
        dtype=tok.dtype,
    )

    A_stack = torch.stack(
        [lora_A[name].weight for name in valid_names],
        dim=0,
    )

    B_stack = torch.stack(
        [lora_B[name].weight for name in valid_names],
        dim=0,
    )

    z = torch.einsum(
        "tf,erf->ter",
        tok,
        A_stack,
    )

    y = torch.einsum(
        "ter,eor->teo",
        z,
        B_stack,
    )

    y = y * scales.view(1, -1, 1)

    delta_flat = torch.einsum(
        "te,teo->to",
        coeff,
        y,
    )

    chosen_global = valid_global[local_idx]

    counts = torch.bincount(
        chosen_global.flatten(),
        minlength=len(self.task_adapter_names),
    ).detach().cpu()

    self.fleet_usage_counts += counts
    self.fleet_tokens_seen += int(tok.shape[0])

    # Canonical adapters use dropout=0.0, so this is Identity in v0.4.
    delta = dropout(delta_flat)

    return delta.view(
        batch,
        *rest,
        delta_flat.size(-1),
    )

def install_masked_router(model):
    patched = 0
    unresolved = []

    for host_name, module in model.named_modules():
        if not hasattr(module, "lora_arrow"):
            continue

        if "arrow_router" not in module.lora_arrow:
            continue

        arrow = module.lora_arrow["arrow_router"]
        semantic_id = resolve_schema_key(host_name)

        if semantic_id is None:
            unresolved.append(host_name)
            continue

        valid = torch.tensor(
            [
                bool(
                    ORIGIN_MASKS[expert].get(
                        semantic_id,
                        False,
                    )
                )
                for expert in EXPERT_ORDER
            ],
            dtype=torch.bool,
        )

        if not valid.any():
            unresolved.append(
                host_name + " EMPTY_MASK"
            )
            continue

        arrow.fleet_valid_expert_mask = valid
        arrow.fleet_schema_key = semantic_id
        arrow.fleet_usage_counts = torch.zeros(
            len(EXPERT_ORDER),
            dtype=torch.long,
        )
        arrow.fleet_tokens_seen = 0

        if not hasattr(
            arrow,
            "_fleet_original_forward",
        ):
            arrow._fleet_original_forward = arrow.forward

        arrow.forward = types.MethodType(
            fleet_masked_arrow_forward,
            arrow,
        )

        patched += 1

    print("patched layers:", patched)

    if unresolved:
        print("unresolved layers:", len(unresolved))
        for x in unresolved[:10]:
            print(" -", x)

    if patched == 0:
        raise RuntimeError(
            "No Arrow layers patched; PEFT internals may have changed."
        )

    return patched

PATCHED_LAYERS = install_masked_router(fleet_model)

# Hard guard: v0.3 would have failed here because recursive had zero eligibility.
eligibility_counts = {
    expert: 0
    for expert in EXPERT_ORDER
}

for _, module in fleet_model.named_modules():
    if not hasattr(module, "lora_arrow"):
        continue

    if "arrow_router" not in module.lora_arrow:
        continue

    arrow = module.lora_arrow["arrow_router"]

    if not hasattr(
        arrow,
        "fleet_valid_expert_mask",
    ):
        continue

    for i, expert in enumerate(EXPERT_ORDER):
        eligibility_counts[expert] += int(
            bool(arrow.fleet_valid_expert_mask[i])
        )

print("eligibility:", eligibility_counts)

dead = [
    expert
    for expert, count in eligibility_counts.items()
    if count == 0
]

if dead:
    raise RuntimeError(
        "Mask bug: zero-eligibility experts: "
        + ", ".join(dead)
    )

print("\\n✅ Every active expert is runtime-eligible.")

## 14. Inspect layer eligibility

In [ ]:
mask_rows = []

for _, module in fleet_model.named_modules():
    if not hasattr(module, "lora_arrow"):
        continue
    if "arrow_router" not in module.lora_arrow:
        continue

    arrow = module.lora_arrow["arrow_router"]

    if not hasattr(
        arrow,
        "fleet_valid_expert_mask",
    ):
        continue

    target = arrow.fleet_schema_key.split(".")[-1]

    row = {"target": target}

    for i, expert in enumerate(EXPERT_ORDER):
        row[expert] = bool(
            arrow.fleet_valid_expert_mask[i]
        )

    mask_rows.append(row)

mask_check_df = (
    pd.DataFrame(mask_rows)
    .drop_duplicates()
    .sort_values("target")
)

print(mask_check_df.to_string(index=False))

eligible_per_target = {}

for target in mask_check_df["target"].unique():
    row = mask_check_df[
        mask_check_df["target"] == target
    ].iloc[0]

    eligible_per_target[target] = sum(
        int(bool(row[expert]))
        for expert in EXPERT_ORDER
    )

print("\neligible experts per target:", eligible_per_target)

if max(eligible_per_target.values()) <= TOP_K:
    print(
        "WARNING: top-k is not selective in any target layer. "
        "Add more eligible experts or reduce TOP_K."
    )
else:
    print(
        "✅ At least one target layer has more eligible experts "
        "than TOP_K, so routing is genuinely selective."
    )

## 15. Generation + routing telemetry

In [ ]:
def reset_route_stats():
    for _, module in fleet_model.named_modules():
        if not hasattr(module, "lora_arrow"):
            continue
        if "arrow_router" not in module.lora_arrow:
            continue

        arrow = module.lora_arrow["arrow_router"]

        if hasattr(
            arrow,
            "fleet_usage_counts",
        ):
            arrow.fleet_usage_counts.zero_()
            arrow.fleet_tokens_seen = 0

def route_stats():
    totals = torch.zeros(
        len(EXPERT_ORDER),
        dtype=torch.long,
    )

    by_target = defaultdict(
        lambda: torch.zeros(
            len(EXPERT_ORDER),
            dtype=torch.long,
        )
    )

    for _, module in fleet_model.named_modules():
        if not hasattr(module, "lora_arrow"):
            continue
        if "arrow_router" not in module.lora_arrow:
            continue

        arrow = module.lora_arrow["arrow_router"]

        if not hasattr(
            arrow,
            "fleet_usage_counts",
        ):
            continue

        counts = arrow.fleet_usage_counts
        totals += counts

        target = arrow.fleet_schema_key.split(".")[-1]
        by_target[target] += counts

    total = int(totals.sum())

    overall = pd.DataFrame([
        {
            "expert": expert,
            "selections": int(totals[i]),
            "share": (
                int(totals[i]) / total
                if total
                else 0.0
            ),
        }
        for i, expert in enumerate(EXPERT_ORDER)
    ])

    target_rows = []

    for target, counts in sorted(by_target.items()):
        subtotal = int(counts.sum())

        for i, expert in enumerate(EXPERT_ORDER):
            target_rows.append({
                "target": target,
                "expert": expert,
                "selections": int(counts[i]),
                "share": (
                    int(counts[i]) / subtotal
                    if subtotal
                    else 0.0
                ),
            })

    return overall, pd.DataFrame(target_rows)

@torch.inference_mode()
def fleet_generate(
    prompt,
    max_new_tokens=180,
):
    fleet_model.set_adapter("arrow_router")

    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        tokenize=True,
        return_dict=True,
        return_tensors="pt",
    )

    device = next(fleet_model.parameters()).device

    inputs = {
        k: v.to(device)
        for k, v in inputs.items()
    }

    out = fleet_model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    return tokenizer.decode(
        out[
            0,
            inputs["input_ids"].shape[-1]:
        ],
        skip_special_tokens=True,
    ).strip()

TEST_PROMPTS = [
    ("math", "Solve carefully: A store sells 8 pens for $12. At the same rate, how much do 14 pens cost?"),
    ("science", "Explain to a student why metals expand when heated."),
    ("humanities", "Explain three key causes of the French Revolution in clear terms."),
    ("language", "Teach the difference between active and passive voice with examples."),
    ("medical", "For education only, explain what insulin does in the body."),
    ("legal_ops", 'Extract a JSON object with governing_law from: "This Agreement is governed by the laws of California."'),
    ("intent", 'Classify the user intent: "Can you move my meeting from Monday to Wednesday?"'),
    ("agent", "Plan how to debug a Python API test that fails only in CI."),
    ("function_calling", 'Given a weather_lookup(city) tool, produce the function call for: "What is the weather in Cebu?"'),
    ("structured", 'Return JSON only with keys "task", "steps", and "risk" for: inspect a failing web scraper.'),
    ("summary", "Summarize in two sentences: A research team built a routing system that combines specialist adapters while keeping a shared language-model backbone."),
    ("prompt", "Follow exactly: answer with three short bullet points about recursion."),
]

all_probe_stats = []

if RUN_GENERATION:
    for i, (expected_capability, prompt) in enumerate(TEST_PROMPTS, 1):
        reset_route_stats()

        answer = fleet_generate(prompt)

        overall, by_target = route_stats()

        print("=" * 100)
        print("TASK", i, "| EXPECTED:", expected_capability)
        print(prompt)
        print("\nOUTPUT")
        print(answer)
        print("\nROUTING SHARE")
        print(overall.to_string(index=False))

        all_probe_stats.append({
            "expected_capability": expected_capability,
            "prompt": prompt,
            "routing": overall.to_dict(orient="records"),
        })
else:
    print("Generation skipped.")

## 15B. Routing diversity diagnostics

With a larger bank, top-2 should actually be selective.

This section checks:
- how many experts ever received a top-k selection,
- whether one expert monopolized the router,
- whether expected specialist prompts change the routing distribution.

In [ ]:
if all_probe_stats:
    aggregate = {
        expert: 0
        for expert in EXPERT_ORDER
    }

    for probe in all_probe_stats:
        for row in probe["routing"]:
            aggregate[row["expert"]] += int(
                row["selections"]
            )

    total = sum(aggregate.values())

    aggregate_df = pd.DataFrame([
        {
            "expert": expert,
            "selections": count,
            "share": count / total if total else 0.0,
        }
        for expert, count in aggregate.items()
    ]).sort_values(
        "selections",
        ascending=False,
    )

    print(
        aggregate_df.to_string(
            index=False
        )
    )

    used_experts = int(
        (aggregate_df["selections"] > 0).sum()
    )

    max_share = float(
        aggregate_df["share"].max()
    )

    print(
        "\nExperts used at least once:",
        used_experts,
        "/",
        len(EXPERT_ORDER),
    )

    print(
        "Largest aggregate routing share:",
        f"{max_share:.2%}",
    )

    if used_experts < min(4, len(EXPERT_ORDER)):
        print(
            "WARNING: router collapsed onto too few experts."
        )

    if max_share > 0.70:
        print(
            "WARNING: one expert dominates >70% of top-k selections."
        )

## 16. Export local artifacts + a license-filtered Hub folder

The local export can contain every compiled adapter you tested.

The Hub-ready folder only copies compiled adapter derivatives whose source repo reports an allowlisted redistributable license.

In [ ]:
EXPORT_ROOT = Path("/content/Fleet-Compiler-v0.5.1")
HUB_UPLOAD_ROOT = Path("/content/Fleet-Compiler-v0.5.1-Hub")

for folder in [EXPORT_ROOT, HUB_UPLOAD_ROOT]:
    if folder.exists():
        shutil.rmtree(folder)
    folder.mkdir(parents=True)

final_overall, final_target = route_stats()

report = {
    "project": "Fleet Compiler",
    "version": "0.5.1-no-training",
    "base_model": BASE_MODEL,
    "training_performed": False,
    "canonical_rank": CANONICAL_RANK,
    "minimum_structural_retention": MIN_STRUCTURAL_RETENTION,
    "dropped_for_retention": dropped_for_retention,
    "active_experts": EXPERT_ORDER,
    "source_adapters": [
        {
            "name": item["name"],
            "description": item["description"],
            "repo": item["repo"],
            "rank": item["rank"],
            "targets": item["targets"],
            "license": item["license"],
            "redistributable": item["redistributable"],
            "active": item["name"] in SOURCE_PAIRS,
        }
        for item in selected_by_name.values()
    ],
    "rank_sweep": rank_sweep_df.to_dict(orient="records"),
    "compile_report": compile_df.to_dict(orient="records"),
    "fidelity": (
        fidelity_df.to_dict(orient="records")
        if len(fidelity_df)
        else []
    ),
    "router": {
        "type": "semantic-key + source-mask Arrow prototype",
        "top_k": TOP_K,
        "patched_layers": PATCHED_LAYERS,
        "eligibility_counts": eligibility_counts,
    },
    "probe_routing": all_probe_stats,
    "last_route_overall": final_overall.to_dict(orient="records"),
    "last_route_by_target": final_target.to_dict(orient="records"),
}

for folder in [EXPORT_ROOT, HUB_UPLOAD_ROOT]:
    (folder / "fleet_report.json").write_text(
        json.dumps(report, indent=2),
        encoding="utf-8",
    )

expert_manifest = {
    "base_model": BASE_MODEL,
    "canonical_rank": CANONICAL_RANK,
    "experts": [
        {
            "name": item["name"],
            "description": item["description"],
            "source_repo": item["repo"],
            "source_license": item["license"],
            "active": item["name"] in SOURCE_PAIRS,
            "included_compiled_artifact": (
                item["name"] in PUBLISHABLE_PACKAGES
            ),
        }
        for item in selected_by_name.values()
    ],
}

(HUB_UPLOAD_ROOT / "expert_manifest.json").write_text(
    json.dumps(expert_manifest, indent=2),
    encoding="utf-8",
)

third_party_lines = [
    "# Third-party adapters",
    "",
    "Fleet Compiler does not claim ownership of source adapters.",
    "Each included compiled derivative remains subject to its source license.",
    "",
]

for item in selected_by_name.values():
    if item["name"] not in SOURCE_PAIRS:
        continue

    third_party_lines.extend([
        f"## {item['name']}",
        "",
        f"- Source: https://huggingface.co/{item['repo']}",
        f"- Reported license: {item['license'] or 'unknown'}",
        f"- Compiled artifact included in Hub folder: {item['name'] in PUBLISHABLE_PACKAGES}",
        "",
    ])

(HUB_UPLOAD_ROOT / "THIRD_PARTY.md").write_text(
    "\n".join(third_party_lines),
    encoding="utf-8",
)

for expert, package in PACKAGES.items():
    shutil.copy2(package, EXPORT_ROOT / package.name)

for expert, package in PUBLISHABLE_PACKAGES.items():
    shutil.copy2(package, HUB_UPLOAD_ROOT / package.name)

model_card = f"""---
library_name: peft
base_model: {BASE_MODEL}
license: other
license_name: Mixed third-party adapter licenses; see THIRD_PARTY.md
tags:
- peft
- lora
- qwen3
- mixture-of-experts
- research
---

# Fleet Compiler v0.5.1

Fleet Compiler is an experimental no-training system for compiling independently
trained LoRA adapters into a shared routing ABI.

This is not a newly pretrained foundation model.

## Runtime

```text
Qwen3-4B-Instruct-2507
+ compiled expert bank
+ original-layer masks
+ top-k token routing
```

## v0.5

- semantic module-key normalization across third-party wrappers
- rank-retention quality gate
- broad multi-capability expert bank\n- curated capability slots
- license/provenance metadata
- hard zero-eligibility guard
- no source-adapter training

See `fleet_report.json` and `expert_manifest.json` for exact sources/results.

## License

Fleet's own code is Apache-2.0. This Hub artifact can contain derivatives of
third-party adapters under different licenses; see THIRD_PARTY.md.
"""

(HUB_UPLOAD_ROOT / "README.md").write_text(
    model_card,
    encoding="utf-8",
)

print("Local export:", EXPORT_ROOT)
print("Hub-ready export:", HUB_UPLOAD_ROOT)

for f in sorted(HUB_UPLOAD_ROOT.iterdir()):
    print(" -", f.name, f"{f.stat().st_size / 1024**2:.2f} MB")

## 17. Optional Hugging Face upload

In [ ]:
PUBLISH_TO_HUB = False
HUB_REPO = "noumenon-labs/Fleet-Compiler-v0.5.1"

if PUBLISH_TO_HUB:
    # First authenticate a write token:
    # !hf auth login

    upload_api = HfApi()

    upload_api.create_repo(
        repo_id=HUB_REPO,
        repo_type="model",
        exist_ok=True,
    )

    upload_api.upload_folder(
        repo_id=HUB_REPO,
        repo_type="model",
        folder_path=str(HUB_UPLOAD_ROOT),
        commit_message="Publish Fleet Compiler v0.5.1",
    )

    print("Uploaded:", f"https://huggingface.co/{HUB_REPO}")
else:
    print(
        "Set PUBLISH_TO_HUB=True only after `hf auth login` "
        "with a write-capable token."
    )

# Publication checklist

A v0.5 run is worth publishing only if:

```text
✅ semantic module normalization passed
✅ ideally 10+ active experts; hard minimum 4 survive config + retention gates
✅ canonical ABI verified
✅ Arrow router created
✅ every active expert has >0 eligible runtime layers
✅ at least one target layer has >TOP_K eligible experts
✅ routing telemetry uses multiple experts
✅ no single expert trivially monopolizes the entire bank
✅ Hub folder excludes unknown/non-allowlisted-license derivatives
```

The point of adding more adapters is not raw count. It is to make top-k routing a real competition among distinct capabilities.